<a href="https://colab.research.google.com/github/Musamehar/ML_Intership/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Constructive Methodology Review of the Research Paper

**Finding 1: "Pages that received a content refresh recovered an average of 25% of their lost traffic within 60 days."**
*   **My Methodology Question:** How did the validation design account for natural seasonality or macro SERP layout changes? I would ask if this was measured using a controlled causal inference method (like a holdout group of declining pages that were *not* refreshed) or if it is purely an observed pre/post-intervention correlation that might be confounded by other variables.

**Finding 2: "Content older than two years exhibits a 40% higher probability of sudden position decay."**
*   **My Methodology Question:** Where does the label come from, and is the time window strictly aligned? I would ask to confirm that the `content_age` was measured strictly *before* the evaluation window where the "position decay" label was observed, ensuring no temporal leakage occurred.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before & After: Naive Random Split vs. Honest Grouped Split
A naive random split mixes URLs from the same client into both training and validation sets. This allows the model to "cheat" by memorizing a specific client's baseline traffic characteristics (domain leakage).

An honest **GroupKFold split on `client_id`** forces the model to predict on completely unseen client domains, reflecting true real-world deployment performance. Below, we run both to observe the performance drop when the model is held to an honest standard.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Setup repository pathing safely
if not os.path.exists('ML_Intership') and not Path('data/raw/content_refresh_anonymized.csv').exists():
    !git clone https://github.com/Musamehar/ML_Intership.git

possible_paths = [
    Path('ML_Intership/data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv')
]

data_path = next((p for p in possible_paths if p.exists()), None)
df = pd.read_csv(data_path)

# 2. Filter qualified slice and define target
df_clean = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()

if 'is_declining_label' in df_clean.columns:
    df_clean['target'] = df_clean['is_declining_label'].astype(int)
else:
    df_clean['target'] = (df_clean['trend_direction'] == 'down').astype(int)

# 3. Build Honest Feature Matrix
df_clean['log_impressions_90d'] = np.log1p(df_clean['impressions_90d'])
df_clean['ctr_90d'] = df_clean['clicks_90d'] / (df_clean['impressions_90d'] + 1e-5)
df_clean['has_keyword'] = (df_clean['word_count'] > 0).astype(int)

feature_cols = ['log_impressions_90d', 'ctr_90d', 'avg_position', 'content_age_days', 'has_keyword']
X = df_clean[feature_cols].fillna(0)
y = df_clean['target']
groups = df_clean['client_id']

rf_model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)

# APPROACH A: The Dishonest "Naive" Random Split
X_train_rand, X_val_rand, y_train_rand, y_val_rand = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model.fit(X_train_rand, y_train_rand)
naive_preds = rf_model.predict_proba(X_val_rand)[:, 1]
naive_auc = roc_auc_score(y_val_rand, naive_preds)

# APPROACH B: The Honest Grouped Split (Client Holdout)
gkf = GroupKFold(n_splits=5)
honest_aucs = []

for train_idx, val_idx in gkf.split(X, y, groups):
    X_train_grp, X_val_grp = X.iloc[train_idx], X.iloc[val_idx]
    y_train_grp, y_val_grp = y.iloc[train_idx], y.iloc[val_idx]

    rf_model.fit(X_train_grp, y_train_grp)
    grp_preds = rf_model.predict_proba(X_val_grp)[:, 1]
    honest_aucs.append(roc_auc_score(y_val_grp, grp_preds))

honest_auc_avg = np.mean(honest_aucs)

print("=" * 65)
print("VALIDATION DESIGN AUDIT: NAIVE VS. HONEST SPLIT")
print("=" * 65)
print(f"1. Naive Random Split ROC-AUC:  {naive_auc:.4f} (Inflated by domain leakage)")
print(f"2. Honest Grouped Split ROC-AUC: {honest_auc_avg:.4f} (True generalizable performance)")
print(f"-> Performance Drop Penalty:     {naive_auc - honest_auc_avg:.4f}")
print("=" * 65)

Cloning into 'ML_Intership'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 129 (delta 40), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.86 MiB | 5.79 MiB/s, done.
Resolving deltas: 100% (40/40), done.
VALIDATION DESIGN AUDIT: NAIVE VS. HONEST SPLIT
1. Naive Random Split ROC-AUC:  0.7423 (Inflated by domain leakage)
2. Honest Grouped Split ROC-AUC: 0.6665 (True generalizable performance)
-> Performance Drop Penalty:     0.0759


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Explicit Leakage Audit Checklist
To ensure the model is discovering real signals and not circular target logic, we enforce the following data contract:
1. **Target Leakage Removed:** The columns `trend_pct` and `trend_direction` are permanently dropped from the feature set, as `is_declining_label` is mathematically derived from them.
2. **Temporal Leakage Prevented:** All features (e.g., `impressions_90d`, `clicks_90d`) are calculated exclusively from the 90-day trailing window observed strictly *prior* to the evaluation of the decay label.
3. **No Product Context Used:** Rule-based outputs like `priority_score` or `health_score` (if present) were excluded to prevent the model from simply memorizing the existing FlyRank logic.

In [2]:
# Programmatic Leakage Test: Check feature correlation to target
# If any feature has a correlation > 0.90, it is a high-risk leakage candidate.

correlations = df_clean[feature_cols + ['target']].corr()['target'].drop('target')
leakage_candidates = correlations[abs(correlations) > 0.85]

print("=" * 55)
print("PROGRAMMATIC LEAKAGE AUDIT")
print("=" * 55)
print("Feature Correlations with Target (checking for > 0.85):")
print(correlations.round(3))
print("-" * 55)
if len(leakage_candidates) == 0:
    print("STATUS: PASSED. No obvious mathematical leakage detected.")
else:
    print(f"WARNING: Potential leakage found in features: {list(leakage_candidates.index)}")

PROGRAMMATIC LEAKAGE AUDIT
Feature Correlations with Target (checking for > 0.85):
log_impressions_90d    0.177
ctr_90d               -0.062
avg_position          -0.029
content_age_days      -0.164
has_keyword            0.090
Name: target, dtype: float64
-------------------------------------------------------
STATUS: PASSED. No obvious mathematical leakage detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Refinement (Moving from Hyped to Honest)

*   **Dishonest / Unsafe Claim:**
    *"Our model accurately predicts exactly which pages will lose traffic. It proves that older pages cause ranking drops, and rewriting these flagged pages will guarantee a recovery in search volume."*

*   **Honest / Public-Safe Claim:**
    *"Our model provides **directional, decision-support scoring** to help SEO editors prioritize their review queues. We **observed** that pages with high historical visibility and higher content age are strongly associated with subsequent traffic decay. This tool effectively surfaces at-risk content, though it does not guarantee causal traffic recovery upon intervention."*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.